In [22]:
from sklearn.linear_model  import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [23]:
# Load saved features
X_combined = np.load("X_combined.npy")

y_pIC50 = np.load("y_pIC50.npy")

print(X_combined.shape)
print(y_pIC50.shape)

(979, 1032)
(979,)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_pIC50, test_size=0.2, random_state=42
)

# Scale features (critical for regression + SVM)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ─── Ridge Regression ─────────────────────────────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

r2_ridge  = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
cv_ridge  = cross_val_score(ridge, X_train_sc, y_train, cv=5, scoring='r2')

print(f"Ridge Regression → R²: {r2_ridge:.3f} | RMSE: {rmse_ridge:.3f}")
print(f"  5-fold CV R²: {cv_ridge.mean():.3f} ± {cv_ridge.std():.3f}")

# ─── Lasso Regression (sparse — forces uninformative bits to zero) ─────────────
lasso = Lasso(alpha=0.01, max_iter=5000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

r2_lasso  = r2_score(y_test, y_pred_lasso)
n_nonzero = np.sum(lasso.coef_ != 0)
print(f"Lasso Regression → R²: {r2_lasso:.3f} | Non-zero features: {n_nonzero}")

# ─── ElasticNet (combines Ridge + Lasso) ──────────────────────────────────────
en = ElasticNet(alpha=0.01, l1_ratio=0.5)
en.fit(X_train_sc, y_train)
r2_en = r2_score(y_test, en.predict(X_test_sc))
print(f"ElasticNet → R²: {r2_en:.3f}")

Ridge Regression → R²: 0.417 | RMSE: 1.141
  5-fold CV R²: 0.392 ± 0.073
Lasso Regression → R²: 0.678 | Non-zero features: 318
ElasticNet → R²: 0.647


In [28]:

# Load processed dataset
df = pd.read_csv("egfr_features_processed.csv")

print(df.head())
print(df.columns)

X_combined = np.load("X_combined.npy")

y_pIC50 = np.load("y_pIC50.npy")

y_active = np.load("y_active.npy")


   action_type  activity_comment  activity_id activity_properties  \
0          NaN               NaN      32260.0                  []   
1          NaN               NaN      32263.0                  []   
2          NaN               NaN      32265.0                  []   
3          NaN               NaN      32267.0                  []   
4          NaN               NaN      32270.0                  []   

  assay_chembl_id                                  assay_description  \
0    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
1    CHEMBL621151  Inhibition of autophosphorylation of human epi...   
2    CHEMBL615325  Inhibition of ligand-induced proliferation in ...   
3    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
4    CHEMBL621151  Inhibition of autophosphorylation of human epi...   

  assay_type  assay_variant_accession  assay_variant_mutation bao_endpoint  \
0          B                      NaN                     NaN  BAO_0000190

In [29]:
from sklearn.svm import SVC, SVR
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, accuracy_score)

In [30]:
# ─── SVM Classifier (Active / Inactive) ────────────────────────────────────────
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combined, y_active, test_size=0.2, random_state=42, stratify=y_active
)
X_train_csc = scaler.fit_transform(X_train_c)
X_test_csc  = scaler.transform(X_test_c)

svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svm_clf.fit(X_train_csc, y_train_c)

y_pred_svm = svm_clf.predict(X_test_csc)
y_prob_svm = svm_clf.predict_proba(X_test_csc)[:, 1]

acc_svm = accuracy_score(y_test_c, y_pred_svm)
auc_svm = roc_auc_score(y_test_c, y_prob_svm)

print(f"SVM Classifier → Accuracy: {acc_svm:.3f} | AUC-ROC: {auc_svm:.3f}")
print(classification_report(y_test_c, y_pred_svm,
                             target_names=['Inactive', 'Active']))

# ─── SVR (Regression — predict pIC50 value) ────────────────────────────────────
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train_sc, y_train)
r2_svr = r2_score(y_test, svr.predict(X_test_sc))
print(f"SVR → R²: {r2_svr:.3f}")

SVM Classifier → Accuracy: 0.908 | AUC-ROC: 0.955
              precision    recall  f1-score   support

    Inactive       0.91      0.89      0.90        93
      Active       0.90      0.92      0.91       103

    accuracy                           0.91       196
   macro avg       0.91      0.91      0.91       196
weighted avg       0.91      0.91      0.91       196

SVR → R²: 0.691
